<a href="https://colab.research.google.com/github/rund-omari/Data.Eng3/blob/main/lectures/Lecture15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from pathlib import Path
import sqlite3
import logging


In [ ]:
raw_bookings = pd.read_csv('/content/SDK_Hotel_Bookings_TEACHING_SAMPLE.csv')

In [ ]:
raw_bookings

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_week_nights,stays_in_weekend_nights,adults,children,babies,adr,market_segment,deposit_type,reservation_status,reservation_status_date
0,Resort Hotel,1,0,2015,January,1,1,2,3,2.0,0,192.04,Corporate,Refundable,Canceled,2015-01-01
1,City Hotel,0,1,2015,February,8,5,3,3,1.0,0,55.39,Online TA,No Deposit,Check-Out,2015-02-08
2,City Hotel,0,4,2015,March,15,3,0,2,0.0,0,160.31,Direct,No Deposit,Check-Out,2015-03-15
3,Resort Hotel,0,10,2015,April,22,3,2,2,NaN,1,153.85,Groups,No Deposit,Check-Out,2015-04-22
4,City Hotel,0,18,2015,May,4,5,2,2,1.0,0,190.85,Groups,No Deposit,Check-Out,2015-05-04
5,City Hotel,0,28,2015,June,11,6,2,2,1.0,0,69.06,Corporate,No Deposit,Check-Out,2015-06-11
6,Resort Hotel,1,29,2015,July,18,4,3,2,2.0,0,185.89,Corporate,Refundable,Canceled,2015-07-18
7,City Hotel,0,30,2015,August,25,5,0,3,0.0,0,181.81,Online TA,Refundable,Check-Out,2015-08-25
8,City Hotel,0,30,2015,September,7,5,3,0,0.0,0,192.07,Online TA,Refundable,Check-Out,2015-09-07
9,Resort Hotel,0,31,2015,October,14,6,3,2,2.0,0,141.77,Groups,Refundable,Check-Out,2015-10-14


In [ ]:
raw_bookings.shape

(48, 16)

In [ ]:
bookings = raw_bookings.assign(total_nights=lambda d: d["stays_in_week_nights"] + d["stays_in_weekend_nights"])

In [ ]:
bookings.shape

(48, 17)

In [ ]:
bookings

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_week_nights,stays_in_weekend_nights,adults,children,babies,adr,market_segment,deposit_type,reservation_status,reservation_status_date,total_nights
0,Resort Hotel,1,0,2015,January,1,1,2,3,2.0,0,192.04,Corporate,Refundable,Canceled,2015-01-01,3
1,City Hotel,0,1,2015,February,8,5,3,3,1.0,0,55.39,Online TA,No Deposit,Check-Out,2015-02-08,8
2,City Hotel,0,4,2015,March,15,3,0,2,0.0,0,160.31,Direct,No Deposit,Check-Out,2015-03-15,3
3,Resort Hotel,0,10,2015,April,22,3,2,2,NaN,1,153.85,Groups,No Deposit,Check-Out,2015-04-22,5
4,City Hotel,0,18,2015,May,4,5,2,2,1.0,0,190.85,Groups,No Deposit,Check-Out,2015-05-04,7
5,City Hotel,0,28,2015,June,11,6,2,2,1.0,0,69.06,Corporate,No Deposit,Check-Out,2015-06-11,8
6,Resort Hotel,1,29,2015,July,18,4,3,2,2.0,0,185.89,Corporate,Refundable,Canceled,2015-07-18,7
7,City Hotel,0,30,2015,August,25,5,0,3,0.0,0,181.81,Online TA,Refundable,Check-Out,2015-08-25,5
8,City Hotel,0,30,2015,September,7,5,3,0,0.0,0,192.07,Online TA,Refundable,Check-Out,2015-09-07,8
9,Resort Hotel,0,31,2015,October,14,6,3,2,2.0,0,141.77,Groups,Refundable,Check-Out,2015-10-14,9


In [ ]:
bookings.to_csv("hotel_transformed_preview.csv", index=False)

##Reusable ETL Functions


In [ ]:
def extract_data(path):
    if not Path(path).is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def transform_data(frame):
    return frame.assign(total_nights=lambda d: d["stays_in_week_nights"] + d["stays_in_weekend_nights"])

def validate_data(frame):
    if "total_nights" not in frame.columns:
        raise ValueError("total_nights is required")
    return frame[frame["total_nights"].gt(0)].copy()

def load_data(frame, output_path):
    frame.to_csv(output_path, index=False)
    return len(frame)

In [ ]:
data = extract_data('/content/SDK_Hotel_Bookings_TEACHING_SAMPLE.csv')

In [ ]:
t_data = transform_data(data)
valid = validate_data(t_data)
print("Loaded data " , load_data(valid, "hotel_preview.csv"))

Loaded data  47


In [ ]:
d_path= '/content/SDK_Hotel_Bookings_TEACHING_SAMPLE.csv'
print("Loaded data " , load_data(validate_data(transform_data(extract_data(d_path))), "hotel_preview.csv"))

INFO | Extracted 48 rows from SDK_Hotel_Bookings_TEACHING_SAMPLE.csv
INFO | Transformed 48 rows
INFO | Validation: 41 valid rows, 7 for review
WARNING | Some hotel records need further investigation


AttributeError: 'tuple' object has no attribute 'groupby'

##SQLite

In [ ]:
hotel_sql = raw_bookings.assign(total_nights=lambda d: d["stays_in_week_nights"] + d["stays_in_weekend_nights"])
with sqlite3.connect("hotel_bookings_demo.db") as conn:
    hotel_sql.to_sql("bookings", conn, if_exists="replace", index=False)
    result = pd.read_sql_query(
        "SELECT hotel, COUNT(*) AS bookings FROM bookings GROUP BY hotel", conn)
print(result)

          hotel  bookings
0          None         1
1    City Hotel        32
2  Resort Hotel        15


##Data Validation

In [ ]:
required = {"hotel", "lead_time", "total_nights", "total_guests"}
missing = required - set(raw_bookings.columns)
missing

{'total_guests', 'total_nights'}

In [ ]:
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

ValueError: Missing required columns: ['total_guests', 'total_nights']

In [ ]:
hotel = raw_bookings.assign(total_nights=lambda d: d["stays_in_week_nights"] + d["stays_in_weekend_nights"])
#total_guests
hotel["total_guests"] = hotel["adults"] + hotel["children"] + hotel["babies"]

In [ ]:
flags = pd.DataFrame(index=hotel.index)
flags["bad_guest_count"] = hotel["total_guests"].isna() | hotel["total_guests"].le(0)
flags["bad_nights"] = hotel["total_nights"].isna() | hotel["total_nights"].le(0)
flags["bad_lead_time"] = hotel["lead_time"].isna() | hotel["lead_time"].lt(0)
flags["bad_adr"] = hotel["adr"].isna() | hotel["adr"].lt(0)
hotel["children_missing"] = hotel["children"].isna() | hotel["children"].eq(0)
flags["missing_children_review"] = hotel["children_missing"]
review_mask = flags.fillna(True).any(axis=1)
valid_bookings = hotel.loc[~review_mask].copy()
records_for_review = hotel.loc[review_mask].copy()
print("Valid:", len(valid_bookings), "Review:", len(records_for_review))

Valid: 21 Review: 27


In [ ]:
records_for_review

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_week_nights,stays_in_weekend_nights,adults,children,babies,adr,market_segment,deposit_type,reservation_status,reservation_status_date,total_nights,total_guests,children_missing
2,City Hotel,0,4,2015,March,15,3,0,2,0.0,0,160.31,Direct,No Deposit,Check-Out,2015-03-15,3,2.0,True
3,Resort Hotel,0,10,2015,April,22,3,2,2,NaN,1,153.85,Groups,No Deposit,Check-Out,2015-04-22,5,NaN,True
7,City Hotel,0,30,2015,August,25,5,0,3,0.0,0,181.81,Online TA,Refundable,Check-Out,2015-08-25,5,3.0,True
8,City Hotel,0,30,2015,September,7,5,3,0,0.0,0,192.07,Online TA,Refundable,Check-Out,2015-09-07,8,0.0,True
12,Resort Hotel,1,60,2015,January,10,0,0,2,2.0,0,180.23,Online TA,No Deposit,Canceled,2015-01-10,0,4.0,False
14,City Hotel,0,89,2015,March,24,6,3,1,0.0,0,112.37,Direct,Refundable,Check-Out,2015-03-24,9,1.0,True
15,Resort Hotel,0,90,2015,April,6,4,1,2,0.0,0,55.03,Online TA,Refundable,Check-Out,2015-04-06,5,2.0,True
17,City Hotel,0,91,2015,June,20,6,2,2,0.0,0,158.01,Corporate,Refundable,Check-Out,2015-06-20,8,2.0,True
18,Resort Hotel,1,105,2015,July,2,6,1,3,0.0,0,146.13,Groups,No Deposit,Canceled,2015-07-02,7,3.0,True
19,City Hotel,0,120,2015,August,9,4,1,2,2.0,0,-20.00,Corporate,No Deposit,Check-Out,2015-08-09,5,4.0,False


##Logging

In [ ]:
logging.basicConfig(level=logging.INFO,format="%(levelname)s | %(message)s",force=True )
logger = logging.getLogger("hotel_etl")
logger.info("Starting Hotel Booking ETL")

INFO | Starting Hotel Booking ETL


In [ ]:
try:
    data = pd.read_csv("/content/SDK_Hotel_Bookings_TEACHING_SAMPLE.csv")
    logger.info("Extracted %d rows", len(data))
    if data.empty:
        logger.warning("Source data is empty")
except FileNotFoundError:
    logger.exception("Missing hotel source file")
    raise  # do not silently continue

INFO | Extracted 48 rows


#Complete Hotel ETL Pipeline


In [ ]:
import logging
import sqlite3
import sys
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("hotel_etl")

REQUIRED = {"hotel", "lead_time", "stays_in_week_nights", "stays_in_weekend_nights","adults", "children", "babies", "adr", "is_canceled"}
NUMERIC = ["lead_time", "stays_in_week_nights", "stays_in_weekend_nights","adults", "children", "babies", "adr"]

In [ ]:
def extract_data(path: str) -> pd.DataFrame:
    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f"Hotel source file was not found: {source}")
    data = pd.read_csv(source)
    logger.info("Extracted %d rows from %s", len(data), source.name)
    return data

In [ ]:
def transform_data(data: pd.DataFrame) -> pd.DataFrame:
    missing = REQUIRED - set(data.columns)
    if missing:
        raise ValueError(f"Required columns missing: {sorted(missing)}")
    result = data.copy()
    for column in NUMERIC:
        result[column] = pd.to_numeric(result[column], errors="coerce")
    result["children_missing"] = result["children"].isna()
    # Business assumption for this teaching example: missing children count = 0.
    # Keep children_missing so questionable rows can be investigated separately.
    result["total_nights"] = (result["stays_in_week_nights"]+ result["stays_in_weekend_nights"])
    result["total_guests"] = (result["adults"] + result["children"].fillna(0) + result["babies"])
    result["estimated_booking_value"] = result["adr"] * result["total_nights"]
    result["lead_time_group"] = pd.cut(
        result["lead_time"],
        bins=[-np.inf, 30, 90, np.inf],
        labels=["Short Notice", "Medium Notice", "Early Booking"],
        right=True,
    )
    result["cancellation_label"] = result["is_canceled"].map(
        {0: "Not Canceled", 1: "Canceled"}
    )
    logger.info("Transformed %d rows", len(result))
    return result

In [ ]:
def validate_data(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    required_derived = {"total_nights", "total_guests", "estimated_booking_value", "lead_time_group"}
    missing = required_derived - set(data.columns)
    if missing:
        raise ValueError(f"Derived columns missing: {sorted(missing)}")
    result = data.copy()
    checks = {
        "missing_hotel": result["hotel"].isna() | result["hotel"].astype("string").str.strip().eq("").fillna(True),
        "invalid_lead_time": result["lead_time"].isna() | result["lead_time"].lt(0),
        "invalid_nights": result["total_nights"].isna() | result["total_nights"].le(0),
        "invalid_guests": result["total_guests"].isna() | result["total_guests"].le(0),
        "invalid_adr": result["adr"].isna() | result["adr"].lt(0),
        "missing_children_review": result["children_missing"],
    }
    flags = pd.DataFrame({name: mask.fillna(True).astype(bool) for name, mask in checks.items()}, index=result.index)
    needs_review = flags.any(axis=1)
    result["needs_review"] = needs_review
    valid = result.loc[~needs_review].copy()
    review = result.loc[needs_review].copy()
    review["review_reason"] = flags.loc[needs_review].apply(
        lambda row: ", ".join(row.index[row.to_numpy()]), axis=1)
    logger.info("Validation: %d valid rows, %d for review", len(valid), len(review))
    if len(review):
        logger.warning("Some hotel records need further investigation")
    return valid, review

In [ ]:
def load_data(valid: pd.DataFrame, review: pd.DataFrame, database_path: str = "hotel_etl.db") -> pd.DataFrame:
    summary = (valid.groupby("hotel", dropna=False).agg(bookings=("hotel", "size"),avg_adr=("adr", "mean"),avg_total_nights=("total_nights", "mean")).reset_index())
    with sqlite3.connect(database_path) as connection:
        valid.to_sql("bookings_clean", connection, if_exists="replace", index=False)
        summary.to_sql("hotel_summary", connection, if_exists="replace", index=False)
        count = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM bookings_clean", connection)["total_rows"].iat[0]
    review.to_csv("hotel_records_for_review.csv", index=False)
    if count != len(valid):
        raise RuntimeError("Database row count does not match validated records")
    logger.info("Loaded %d bookings to SQLite and exported %d review rows", count, len(review))
    return summary

In [ ]:
def run_pipeline(source: str) -> pd.DataFrame:
    logger.info("Hotel ETL pipeline started")
    try:
        raw = extract_data(source)
        transformed = transform_data(raw)
        valid, review = validate_data(transformed)
        summary = load_data(valid, review)
        logger.info("Pipeline completed successfully")
        return summary
    except Exception:
        logger.exception("Hotel ETL pipeline failed")
        raise

In [ ]:
d_path= '/content/SDK_Hotel_Bookings_TEACHING_SAMPLE.csv'
print(run_pipeline(d_path).to_string(index=False))

INFO | Hotel ETL pipeline started
INFO | Extracted 48 rows from SDK_Hotel_Bookings_TEACHING_SAMPLE.csv
INFO | Transformed 48 rows
INFO | Validation: 41 valid rows, 7 for review
WARNING | Some hotel records need further investigation
INFO | Loaded 41 bookings to SQLite and exported 7 review rows
INFO | Pipeline completed successfully


       hotel  bookings    avg_adr  avg_total_nights
  City Hotel        30 140.240000          5.533333
Resort Hotel        11 338.499091          5.636364
